# Math primer for numerical optimization

This notebook is a **hands-on, code-first refresher** on the linear algebra and calculus operations that show up constantly in numerical optimization.

It includes:
- **NumPy** for concrete computations
- **JAX** for *automatic differentiation* (grad/Jacobian/Hessian) and vectorization (`vmap`)
- **PyTorch** for built-in autodiff (including higher-order derivatives)

Throughout, we emphasize:
- shapes & conventions (`(n,)` vs `(n,1)`)
- sanity checks (finite differences / gradient checking)
- optimization-relevant primitives (projections, least squares, PSD matrices, conditioning, etc.)

> Tip: Optimization bugs are often **shape bugs** or **sign bugs**. Keep printing shapes and writing small checks.

---

In [1]:

import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

print("NumPy version:", np.__version__)

NumPy version: 1.25.2


## 0. Optional imports: JAX and PyTorch

This notebook is designed to run even if **JAX** or **PyTorch** are not installed.
- If JAX is missing, JAX cells will print a helpful message and skip execution.
- Same for PyTorch.

If you want to install locally:
- **JAX (CPU)**: `pip install "jax[cpu]"`  
- **PyTorch (CPU)**: follow the instructions at pytorch.org

(Installation commands are not executed automatically here.)

In [3]:

# JAX (optional)
try:
    import jax
    import jax.numpy as jnp
    from jax import grad, jacfwd, jacrev, hessian, vmap, jit
    JAX_AVAILABLE = True
    print("JAX version:", jax.__version__)
except Exception as e:
    JAX_AVAILABLE = False
    print("JAX not available:", repr(e))

# PyTorch (optional)
try:
    import torch
    TORCH_AVAILABLE = True
    print("PyTorch version:", torch.__version__)
except Exception as e:
    TORCH_AVAILABLE = False
    print("PyTorch not available:", repr(e))

JAX version: 0.4.38
PyTorch version: 2.8.0


## 1. Vectors and matrices: shapes, dtype, and broadcasting

Optimization code often mixes:
- scalars: shape `()`
- vectors: shape `(n,)` (1D) or `(n,1)` (column vector)
- matrices: shape `(m,n)`

In NumPy, a 1D array has no orientation; you choose it via usage (`x @ y` vs `x[:,None]`).

In [5]:
n = 5
x = rng.normal(size=(n,))        # 1D vector
X_col = x[:, None]               # explicit column vector shape (n,1)
A = rng.normal(size=(n, n))      # square matrix

print("x shape:", x.shape)
print("X_col shape:", X_col.shape)
print("A shape:", A.shape)

# Broadcasting example: adding a scalar and a vector
print("x + 1.0:", x + 1.0)

# Broadcasting example: adding a column vector to a row vector gives a matrix
row = rng.normal(size=(1, n))
print("X_col + row has shape:", (X_col + row).shape)

x shape: (5,)
X_col shape: (5, 1)
A shape: (5, 5)
x + 1.0: [1.1257 0.8679 1.6404 1.1049 0.4643]
X_col + row has shape: (5, 5)


### Beyond basics: `reshape`, `ravel`, and views vs copies

In performance-sensitive optimization, unnecessary copies matter.  
- `reshape` often returns a **view** when possible
- `ravel` returns a view when possible; `flatten` always copies

In [7]:

y = np.arange(12)
Y = y.reshape(3, 4)  # often a view
print("Y:\n", Y)
print("Y.base is y?", Y.base is y)

y_ravel = Y.ravel()
y_flatten = Y.flatten()
print("ravel shares memory?", y_ravel.base is not None)
print("flatten shares memory?", y_flatten.base is not None)

Y:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
Y.base is y? True
ravel shares memory? True
flatten shares memory? False


## 2. Dot products, outer products, and matrix products

- **Dot / inner product**:  \(\langle x, y \rangle = x^\top y\)
- **Outer product**: \(x y^\top\) (rank-1 matrix)
- **Matrix-vector**: \(Ax\)
- **Matrix-matrix**: \(AB\)

Use `@` for linear algebra; use `*` for elementwise multiplication.

In [9]:

x = rng.normal(size=(n,))
y = rng.normal(size=(n,))
A = rng.normal(size=(n, n))
B = rng.normal(size=(n, n))

dot_xy = x @ y                  # scalar
outer_xy = np.outer(x, y)       # (n,n)
Ax = A @ x                      # (n,)
AB = A @ B                      # (n,n)

print("x @ y =", dot_xy)
print("outer(x,y) shape:", outer_xy.shape)
print("A @ x shape:", Ax.shape)
print("A @ B shape:", AB.shape)

# Elementwise operations
print("Elementwise x*y shape:", (x * y).shape)

x @ y = -0.604252762088693
outer(x,y) shape: (5, 5)
A @ x shape: (5,)
A @ B shape: (5, 5)
Elementwise x*y shape: (5,)


### Einstein summation (`einsum`) for clarity

`einsum` is handy when you want to specify tensor contractions explicitly.

Examples:
- dot: `np.einsum("i,i->", x, y)`
- matrix-vector: `np.einsum("ij,j->i", A, x)`
- quadratic form: `np.einsum("i,ij,j->", x, A, x)` gives \(x^\top A x\)

In [11]:

dot_einsum = np.einsum("i,i->", x, y)
Ax_einsum = np.einsum("ij,j->i", A, x)
quad = np.einsum("i,ij,j->", x, A, x)

print("dot_einsum:", dot_einsum, " (matches x@y:", np.allclose(dot_einsum, x@y), ")")
print("Ax_einsum matches A@x:", np.allclose(Ax_einsum, A@x))
print("quadratic form x^T A x:", quad)

dot_einsum: -0.604252762088693  (matches x@y: True )
Ax_einsum matches A@x: True
quadratic form x^T A x: -4.399293791452673


## 3. Vector norms and induced matrix norms

Common vector norms:
- \(\|x\|_1 = \sum_i |x_i|\)
- \(\|x\|_2 = \sqrt{\sum_i x_i^2}\) (Euclidean norm)
- \(\|x\|_\infty = \max_i |x_i|\)

In optimization, norms show up in:
- regularization (L1 / L2)
- step sizes and stopping criteria

In [14]:

x = rng.normal(size=(n,))
A = rng.normal(size=(n, n))

l1 = np.linalg.norm(x, ord=1)
l2 = np.linalg.norm(x, ord=2)
linf = np.linalg.norm(x, ord=np.inf)

print("x:", x)
print("||x||_1 =", l1)
print("||x||_2 =", l2)
print("||x||_inf =", linf)

# Normalize a vector (safe pattern)
x_unit = x / (np.linalg.norm(x) + 1e-12)
print("||x_unit||_2:", np.linalg.norm(x_unit))

x: [ 1.0315  0.161  -0.5855 -1.3412 -1.4015]
||x||_1 = 4.520731414702251
||x||_2 = 2.27942817132064
||x||_inf = 1.401520214917428
||x_unit||_2: 0.9999999999995611


### Cosine similarity and angles

For nonzero vectors,
\[
\cos(\theta) = \frac{x^\top y}{\|x\|_2 \|y\|_2}.
\]

This is used in optimization heuristics (e.g., detecting alignment between gradients).

In [16]:

x = rng.normal(size=(n,))
y = rng.normal(size=(n,))

cos_theta = (x @ y) / ((np.linalg.norm(x) + 1e-12) * (np.linalg.norm(y) + 1e-12))
theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))

print("cos(theta) =", cos_theta)
print("theta (radians) =", theta)

cos(theta) = -0.06376972846178836
theta (radians) = 1.6346093553062677


## 4. Projections and least squares (core optimization primitive)

**Projection onto a vector** \(u \neq 0\):
\[
\mathrm{proj}_u(x) = \frac{u^\top x}{u^\top u}\, u.
\]

**Least squares** solves:
\[
\min_x \|Ax - b\|_2^2.
\]

In optimization, least squares appears everywhere:
- linear regression
- Gauss–Newton and Levenberg–Marquardt
- subproblems inside constrained optimization

In [18]:

m, n = 8, 3
A = rng.normal(size=(m, n))
x_true = rng.normal(size=(n,))
b = A @ x_true + 0.01 * rng.normal(size=(m,))  # nearly consistent system

# Solve least squares via QR/SVD under the hood
x_ls, residuals, rank, svals = np.linalg.lstsq(A, b, rcond=None)

print("x_true:", x_true)
print("x_ls  :", x_ls)
print("||A x_ls - b||_2:", np.linalg.norm(A @ x_ls - b))
print("rank:", rank)

x_true: [-2.3982  0.5131 -0.2976]
x_ls  : [-2.3973  0.5128 -0.2995]
||A x_ls - b||_2: 0.029897429185125925
rank: 3


### Avoid explicit matrix inverse

For linear systems \(Ax=b\), **prefer** `solve` over `inv`:
- `x = np.linalg.solve(A, b)` is more stable and faster
- `np.linalg.inv(A) @ b` forms the inverse explicitly (worse numerically)

This matters a lot in optimization algorithms (Newton, quasi-Newton, KKT systems).

In [20]:

n = 5
A = rng.normal(size=(n, n))
A = A.T @ A + 1e-2 * np.eye(n)  # make it SPD-ish
b = rng.normal(size=(n,))

x_solve = np.linalg.solve(A, b)
x_inv = np.linalg.inv(A) @ b

print("Difference between solve and inv:", np.linalg.norm(x_solve - x_inv))

Difference between solve and inv: 2.0350724194510405e-15


## 5. Conditioning and why it matters

The **condition number** (2-norm) of a matrix is:
\[
\kappa_2(A) = \|A\|_2 \|A^{-1}\|_2 = \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}.
\]

- Large \(\kappa\) means the problem is *ill-conditioned*.
- Ill-conditioning can cause:
  - slow convergence of gradient descent
  - numerical instability in Newton/least squares
  - sensitivity to noise and rounding

Let's see this with a toy example.

In [22]:

# Create a matrix with a controlled spectrum via SVD
U, _ = np.linalg.qr(rng.normal(size=(6, 6)))
V, _ = np.linalg.qr(rng.normal(size=(6, 6)))
singular_values = np.array([1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5])
A = U @ np.diag(singular_values) @ V.T

cond = np.linalg.cond(A)
print("cond_2(A) ~", cond)

# Perturb b slightly and see solution sensitivity
b = rng.normal(size=(6,))
x = np.linalg.solve(A, b)

eps = 1e-6
b_pert = b + eps * rng.normal(size=b.shape)
x_pert = np.linalg.solve(A, b_pert)

print("Relative change in b:", np.linalg.norm(b_pert - b) / np.linalg.norm(b))
print("Relative change in x:", np.linalg.norm(x_pert - x) / np.linalg.norm(x))

cond_2(A) ~ 99999.9999997035
Relative change in b: 1.0554270589695626e-06
Relative change in x: 5.802499053632092e-06


## 6. Quadratic forms, PSD matrices, and convexity

A quadratic function
\[
f(x) = \tfrac12 x^\top Q x + c^\top x + d
\]
is **convex** iff \(Q \succeq 0\) (positive semidefinite).

Useful facts:
- If \(Q\) is symmetric, then \(x^\top Q x\) is a scalar quadratic form.
- \(Q \succeq 0\) iff all eigenvalues of \(Q\) are nonnegative.

In [24]:

n = 4
M = rng.normal(size=(n, n))
Q = M.T @ M  # PSD by construction
c = rng.normal(size=(n,))
d = 0.3

def f_quad(x):
    return 0.5 * x @ Q @ x + c @ x + d

x = rng.normal(size=(n,))
print("f(x) =", f_quad(x))

eigvals = np.linalg.eigvalsh(Q)
print("eigenvalues(Q):", eigvals)
print("Q is PSD?", np.all(eigvals >= -1e-12))

# Quadratic form is always >= 0 for PSD Q (up to numeric noise)
for _ in range(3):
    z = rng.normal(size=(n,))
    print("z^T Q z =", z @ Q @ z)

f(x) = 3.383058426360024
eigenvalues(Q): [ 0.0422  2.0382  4.2509 13.5013]
Q is PSD? True
z^T Q z = 2.0539800908979617
z^T Q z = 38.7998778049022
z^T Q z = 10.891809980169368


### Gradient and Hessian of a quadratic (manual derivation)

For symmetric \(Q\),
\[
\nabla f(x) = Qx + c,\qquad \nabla^2 f(x) = Q.
\]

This is the simplest “optimization test function” because you can check everything exactly.

In [26]:

def grad_quad_manual(x):
    return Q @ x + c

def hess_quad_manual(x):
    return Q

x = rng.normal(size=(n,))
print("grad manual:", grad_quad_manual(x))
print("hess manual (first row):", hess_quad_manual(x)[0])

grad manual: [ 1.9283  1.9689 -2.2011 -2.1673]
hess manual (first row): [ 9.5248 -0.4136  3.6468  3.9792]


## 7. Finite differences (gradient checking)

Even with autodiff, **finite differences** are a great debugging tool.

For a scalar function \(f:\mathbb{R}^n\to\mathbb{R}\),
- Forward difference (cheap, less accurate):
  \[
  \frac{\partial f}{\partial x_i}\approx \frac{f(x + \epsilon e_i) - f(x)}{\epsilon}
  \]
- Central difference (more accurate):
  \[
  \frac{\partial f}{\partial x_i}\approx \frac{f(x + \epsilon e_i) - f(x - \epsilon e_i)}{2\epsilon}
  \]

We'll implement central differences and compare to the manual quadratic gradient.

In [28]:

def grad_fd_central(f, x, eps=1e-6):
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(x.size):
        e = np.zeros_like(x)
        e[i] = 1.0
        g[i] = (f(x + eps * e) - f(x - eps * e)) / (2 * eps)
    return g

x = rng.normal(size=(n,))
g_manual = grad_quad_manual(x)
g_fd = grad_fd_central(f_quad, x, eps=1e-6)

print("||g_manual - g_fd||:", np.linalg.norm(g_manual - g_fd))
print("g_manual:", g_manual)
print("g_fd    :", g_fd)

||g_manual - g_fd||: 1.221851985939198e-09
g_manual: [10.3986 -0.5839  5.9523 -0.9517]
g_fd    : [10.3986 -0.5839  5.9523 -0.9517]


## 8. Jacobians, directional derivatives, and the chain rule

In optimization you often have vector-valued functions:
- constraints \(c(x)\in\mathbb{R}^m\)
- residuals \(r(x)\in\mathbb{R}^m\) (least squares)

The **Jacobian** \(J(x)\in\mathbb{R}^{m\times n}\) contains partial derivatives:
\[
J_{ij} = \frac{\partial r_i}{\partial x_j}.
\]

A **directional derivative** of scalar \(f\) along direction \(v\) is:
\[
D_v f(x) = \nabla f(x)^\top v.
\]

In [ ]:

# Example residual r(x) = [sin(x0) + x1^2,  x0*x1]
def r_np(x):
    x0, x1 = x
    return np.array([np.sin(x0) + x1**2, x0 * x1], dtype=float)

def jac_fd_central(r, x, eps=1e-6):
    x = np.asarray(x, dtype=float)
    m = r(x).size
    n = x.size
    J = np.zeros((m, n), dtype=float)
    for j in range(n):
        e = np.zeros_like(x)
        e[j] = 1.0
        J[:, j] = (r(x + eps*e) - r(x - eps*e)) / (2*eps)
    return J

x = np.array([0.3, -0.7])
J_fd = jac_fd_central(r_np, x)
print("r(x) =", r_np(x))
print("Jacobian (finite diff):\n", J_fd)

# Directional derivative of f(x)=||r(x)||^2 along v
def f_np(x):
    rr = r_np(x)
    return rr @ rr

g_fd = grad_fd_central(f_np, x)
v = np.array([1.0, 2.0])
dir_deriv = g_fd @ v
print("Directional derivative approx:", dir_deriv)

## 9. JAX autodiff: gradients, Jacobians, and Hessians

JAX provides composable automatic differentiation:
- `grad(f)` for scalar-output functions
- `jacfwd`, `jacrev` for Jacobians (forward/reverse mode)
- `hessian(f)` for Hessians of scalar functions

We'll redo the quadratic example in JAX and verify:
- JAX gradient matches manual gradient
- JAX Hessian equals \(Q\)

In [30]:

if not JAX_AVAILABLE:
    print("Skipping: JAX not installed.")
else:
    key = jax.random.PRNGKey(0)
    n = 4
    M = jax.random.normal(key, (n, n))
    Q = M.T @ M
    c = jax.random.normal(key, (n,))
    d = 0.3

    def f(x):
        return 0.5 * x @ Q @ x + c @ x + d

    g = grad(f)
    H = hessian(f)

    x = jax.random.normal(key, (n,))
    g_jax = g(x)
    H_jax = H(x)

    g_manual = Q @ x + c

    print("||g_jax - g_manual||:", jnp.linalg.norm(g_jax - g_manual))
    print("||H_jax - Q||:", jnp.linalg.norm(H_jax - Q))
    print("Hessian symmetry check:", jnp.max(jnp.abs(H_jax - H_jax.T)))

||g_jax - g_manual||: 2.4575624e-07
||H_jax - Q||: 0.0
Hessian symmetry check: 0.0


### JAX: Jacobian of a vector function and choosing `jacfwd` vs `jacrev`

- `jacfwd` uses **forward-mode** (good when input dim is small).
- `jacrev` uses **reverse-mode** (good when output dim is small).

We'll compute the Jacobian of the same residual function \(r(x)\).

In [32]:

if not JAX_AVAILABLE:
    print("Skipping: JAX not installed.")
else:
    def r(x):
        x0, x1 = x
        return jnp.array([jnp.sin(x0) + x1**2, x0 * x1])

    x = jnp.array([0.3, -0.7])

    J_fwd = jacfwd(r)(x)
    J_rev = jacrev(r)(x)

    print("Jacobian (forward mode):\n", np.array(J_fwd))
    print("Jacobian (reverse mode):\n", np.array(J_rev))
    print("Difference:", np.max(np.abs(J_fwd - J_rev)))

Jacobian (forward mode):
 [[ 0.9553 -1.4   ]
 [-0.7     0.3   ]]
Jacobian (reverse mode):
 [[ 0.9553 -1.4   ]
 [-0.7     0.3   ]]
Difference: 0.0


### JAX: `vmap` for batched computations

Optimization often needs evaluating functions/gradients on a *batch*:
- stochastic optimization (mini-batches)
- evaluating many starting points
- Monte Carlo estimators

`vmap` vectorizes a function over a leading batch dimension without Python loops.

In [34]:

if not JAX_AVAILABLE:
    print("Skipping: JAX not installed.")
else:
    def rosenbrock(x):
        # x is shape (2,)
        a, b = 1.0, 100.0
        return (a - x[0])**2 + b * (x[1] - x[0]**2)**2

    grad_rosen = grad(rosenbrock)

    X = jax.random.normal(jax.random.PRNGKey(1), (10, 2))  # batch of 10 points
    G = vmap(grad_rosen)(X)  # shape (10,2)

    print("X shape:", X.shape)
    print("G shape:", G.shape)
    print("First point:", np.array(X[0]))
    print("Grad at first point:", np.array(G[0]))

X shape: (10, 2)
G shape: (10, 2)
First point: [-0.9859 -1.0067]
Grad at first point: [-784.3036 -395.7388]


### JAX: `jit` for speed (optional)

`jit` compiles functions with XLA. For teaching and debugging, you can ignore it.
For larger computations, it can be a big win.

We show it here because optimization workloads are often iterative.

In [36]:

if not JAX_AVAILABLE:
    print("Skipping: JAX not installed.")
else:
    rosen_jit = jit(rosenbrock)
    grad_rosen_jit = jit(grad(rosenbrock))

    x = jnp.array([1.2, 1.2])
    print("rosenbrock(x):", float(rosen_jit(x)))
    print("grad rosenbrock(x):", np.array(grad_rosen_jit(x)))

rosenbrock(x): 5.8000030517578125
grad rosenbrock(x): [115.6 -48. ]


## 10. PyTorch autodiff: gradients, Jacobians, Hessians

PyTorch uses a dynamic computation graph and provides:
- `.backward()` for scalar losses
- `torch.autograd.grad` for more control
- `torch.autograd.functional.jacobian` / `hessian` for higher-order derivatives

We'll reproduce the quadratic function and check gradient/Hessian.

In [39]:

if not TORCH_AVAILABLE:
    print("Skipping: PyTorch not installed.")
else:
    torch.manual_seed(0)

    n = 4
    M = torch.randn(n, n)
    Q = M.T @ M
    c = torch.randn(n)
    d = 0.3

    def f_torch(x):
        return 0.5 * x @ Q @ x + c @ x + d

    x = torch.randn(n, requires_grad=True)
    fx = f_torch(x)
    fx.backward()  # populates x.grad

    g_torch = x.grad
    g_manual = Q @ x.detach() + c

    print("f(x):", float(fx))
    print("||g_torch - g_manual||:", float(torch.norm(g_torch - g_manual)))

    # Hessian via autograd.functional
    H = torch.autograd.functional.hessian(f_torch, x.detach())
    print("||H - Q||:", float(torch.norm(H - Q)))

f(x): 3.151822805404663
||g_torch - g_manual||: 2.6656007889869215e-07
||H - Q||: 0.0


/var/folders/m7/3rglclls3gs685x7cpg2fpt0hwg1yd/T/ipykernel_12542/446407453.py:22: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("f(x):", float(fx))


### PyTorch: Jacobian of a vector function

We'll compute Jacobian of the residual function \(r(x)\) again.

In [41]:

if not TORCH_AVAILABLE:
    print("Skipping: PyTorch not installed.")
else:
    def r_torch(x):
        x0, x1 = x[0], x[1]
        return torch.stack([torch.sin(x0) + x1**2, x0 * x1])

    x = torch.tensor([0.3, -0.7], dtype=torch.float32)
    J = torch.autograd.functional.jacobian(r_torch, x)
    print("Jacobian:\n", J)

Jacobian:
 tensor([[ 0.9553, -1.4000],
        [-0.7000,  0.3000]])


### Autodiff gotchas in optimization code

Common pitfalls:
- Forgetting to zero gradients: `optimizer.zero_grad()` or `x.grad.zero_()`
- Accumulating gradients unintentionally across iterations
- Using in-place ops that break autograd (PyTorch warns/errors sometimes)
- Mixing `detach()` / `.item()` too early, losing gradients
- Using `with torch.no_grad()` carefully when updating parameters

We'll do a tiny gradient descent loop correctly.

In [43]:

if not TORCH_AVAILABLE:
    print("Skipping: PyTorch not installed.")
else:
    # Minimize Rosenbrock with vanilla gradient descent (for demonstration only)
    def rosen_torch(x):
        a, b = 1.0, 100.0
        return (a - x[0])**2 + b * (x[1] - x[0]**2)**2

    x = torch.tensor([1.2, 1.2], dtype=torch.float32, requires_grad=True)
    lr = 1e-3

    for k in range(10):
        if x.grad is not None:
            x.grad.zero_()     # important!

        loss = rosen_torch(x)
        loss.backward()

        with torch.no_grad():
            x -= lr * x.grad

        print(f"iter {k:02d}  loss={float(loss):.6f}  x={x.detach().numpy()}")

iter 00  loss=5.800000  x=[1.0844 1.248 ]
iter 01  loss=0.526626  x=[1.1155 1.2336]
iter 02  loss=0.024884  x=[1.1105 1.2357]
iter 03  loss=0.012875  x=[1.1114 1.2352]
iter 04  loss=0.012410  x=[1.1112 1.2352]
iter 05  loss=0.012385  x=[1.1112 1.2351]
iter 06  loss=0.012376  x=[1.1111 1.235 ]
iter 07  loss=0.012368  x=[1.1111 1.235 ]
iter 08  loss=0.012360  x=[1.1111 1.2349]
iter 09  loss=0.012351  x=[1.111  1.2348]


### JAX/PyTorch Hessian for Rosenbrock (exact via autodiff)

Finite-difference Hessians can be noisy. Autodiff gives exact derivatives up to floating-point error.

In [ ]:

if not JAX_AVAILABLE:
    print("Skipping: JAX not installed.")
else:
    x0 = jnp.array([1.2, 1.2])
    g0 = grad(rosenbrock)(x0)
    H0 = hessian(rosenbrock)(x0)
    p = jnp.linalg.solve(H0, -g0)
    x1 = x0 + p
    print("JAX  f(x0):", float(rosenbrock(x0)))
    print("JAX  ||g0||:", float(jnp.linalg.norm(g0)))
    print("JAX  f(x1):", float(rosenbrock(x1)))

In [ ]:

if not TORCH_AVAILABLE:
    print("Skipping: PyTorch not installed.")
else:
    def rosen_torch_vec(x):
        a, b = 1.0, 100.0
        return (a - x[0])**2 + b * (x[1] - x[0]**2)**2

    x0 = torch.tensor([1.2, 1.2], dtype=torch.float64, requires_grad=True)
    g0 = torch.autograd.grad(rosen_torch_vec(x0), x0, create_graph=True)[0]
    H0 = torch.autograd.functional.hessian(rosen_torch_vec, x0)
    p = torch.linalg.solve(H0, -g0.detach())
    x1 = x0.detach() + p
    print("Torch f(x0):", float(rosen_torch_vec(x0).detach()))
    print("Torch ||g0||:", float(torch.linalg.vector_norm(g0.detach())))
    print("Torch f(x1):", float(rosen_torch_vec(x1).detach()))